In [ ]:
import sys, os
# Add src/ to path so all module imports work
sys.path.insert(0, os.path.abspath("../src"))


In [ ]:
# === Resumen por bin + gráficas (barras y 3D) ===
import os
)

from deeppack3d import deeppack3d
import math
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# ---- Parámetros de corrida ----
METHOD = 'bl'              # <--- CAMBIADO: antes era 'rl'
K = 40
DATA_MODE = 'file'         # usamos tu archivo de datos
PATH = "../data/input_worst_case.txt"       # asegúrate de que este archivo exista ahí
N_ITERS = 200              # solo se usa si DATA_MODE == 'generated'
SEED = 42                  # solo se usa si DATA_MODE == 'generated'

# (Opcional) tamaño del contenedor/pallet (W,H,D)
CONTAINER_SIZE = None      # None => se infiere por bounding box

# ---- Recolectar colocaciones por bin ----
bins = []               # lista de bins; cada bin es una lista de items
current_bin_items = None

gen_kwargs = dict(data=DATA_MODE, verbose=0)
if DATA_MODE == 'file':
    gen_kwargs['path'] = PATH
else:
    gen_kwargs['n_iterations'] = N_ITERS
    gen_kwargs['seed'] = SEED

for result in deeppack3d(METHOD, K, **gen_kwargs):
    if result is None:
        # empieza un nuevo bin
        current_bin_items = []
        bins.append(current_bin_items)
        continue
    bin_id, (x,y,z), (w,h,d), meta = result
    if current_bin_items is None:
        current_bin_items = []
        bins.append(current_bin_items)
    current_bin_items.append((x,y,z,w,h,d))

num_bins = len(bins)
print(f"🧱 Bins detectados: {num_bins}")

# ---- Utilidad por bin ----
def bin_utilization(items, container_size=None):
    """Devuelve (utilizacion, used_volume, container_volume, (W,H,D) usado)"""
    if not items:
        return 0.0, 0.0, 0.0, (0,0,0)
    used_volume = sum(w*h*d for (_,_,_,w,h,d) in items)
    if container_size is None:
        # inferir por bounding box de lo empacado
        max_x = max(x+w for (x,y,z,w,h,d) in items)
        max_y = max(y+h for (x,y,z,w,h,d) in items)
        max_z = max(z+d for (x,y,z,w,h,d) in items)
        W,H,D = max_x, max_y, max_z
    else:
        W,H,D = container_size
    container_volume = max(1, W*H*D)  # evita div/0
    return (used_volume / container_volume, used_volume, container_volume, (W,H,D))

utilizations = []
sizes_used = []
for i, items in enumerate(bins):
    u, used_v, cont_v, sz = bin_utilization(items, CONTAINER_SIZE)
    utilizations.append(u)
    sizes_used.append(sz)
    print(f"Bin {i}: util={u:.3f}  used_vol={used_v:.0f}  cont_vol={cont_v:.0f}  size={sz}")

# ---- Gráfico de barras: utilización por bin ----
plt.figure(figsize=(8,4))
plt.bar(range(len(utilizations)), utilizations)
plt.xlabel("Bin")
plt.ylabel("Utilización (0–1)")
plt.title(f"Utilización por bin — {METHOD.upper()}  k={K}")
plt.ylim(0, 1.0)
plt.show()

# ---- 3D del bin que elijas ----
BIN_TO_PLOT = 40  # cambia el índice para ver otro bin

def draw_cuboid(ax, x, y, z, w, h, d, alpha=0.35):
    # Si por algún error vino una caja "vacía", no la dibujes
    if w <= 0 or h <= 0 or d <= 0:
        return

    X = [x, x+w]; Y = [y, y+h]; Z = [z, z+d]
    faces = [
        [(X[0],Y[0],Z[0]), (X[1],Y[0],Z[0]), (X[1],Y[1],Z[0]), (X[0],Y[1],Z[0])],
        [(X[0],Y[0],Z[1]), (X[1],Y[0],Z[1]), (X[1],Y[1],Z[1]), (X[0],Y[1],Z[1])],
        [(X[0],Y[0],Z[0]), (X[1],Y[0],Z[0]), (X[1],Y[0],Z[1]), (X[0],Y[0],Z[1])],
        [(X[0],Y[1],Z[0]), (X[1],Y[1],Z[0]), (X[1],Y[1],Z[1]), (X[0],Y[1],Z[1])],
        [(X[0],Y[0],Z[0]), (X[0],Y[1],Z[0]), (X[0],Y[1],Z[1]), (X[0],Y[0],Z[1])],
        [(X[1],Y[0],Z[0]), (X[1],Y[1],Z[0]), (X[1],Y[1],Z[1]), (X[1],Y[0],Z[1])],
    ]
    poly = Poly3DCollection(faces, edgecolor='k', alpha=alpha)
    ax.add_collection3d(poly)

def get_axis_limits(items, fallback=(1,1,1)):
    if not items:
        return (0,1),(0,1),(0,1)
    max_x = max(x+w for (x,y,z,w,h,d) in items)
    max_y = max(y+h for (x,y,z,w,h,d) in items)
    max_z = max(z+d for (x,y,z,w,h,d) in items)
    return (0, max(max_x, fallback[0])), (0, max(max_y, fallback[1])), (0, max(max_z, fallback[2]))

items = bins[BIN_TO_PLOT] if BIN_TO_PLOT < num_bins else []
(W,H,D) = sizes_used[BIN_TO_PLOT] if BIN_TO_PLOT < num_bins else (1,1,1)

fig = plt.figure(figsize=(6,6))
ax = fig.add_subplot(111, projection='3d')

# dibuja contenedor solo con líneas (sin Poly3DCollection para evitar el bug)
def draw_container_wire(ax, W,H,D):
    edges = [
        ((0,0,0),(W,0,0)), ((W,0,0),(W,H,0)),
        ((W,H,0),(0,H,0)), ((0,H,0),(0,0,0)),
        ((0,0,D),(W,0,D)), ((W,0,D),(W,H,D)),
        ((W,H,D),(0,H,D)), ((0,H,D),(0,0,D)),
        ((0,0,0),(0,0,D)), ((W,0,0),(W,0,D)),
        ((W,H,0),(W,H,D)), ((0,H,0),(0,H,D)),
    ]
    for (x1,y1,z1),(x2,y2,z2) in edges:
        ax.plot([x1,x2],[y1,y2],[z1,z2], linewidth=1.2)

# dibuja contenedor inferido o indicado
draw_container_wire(ax, W,H,D)

# dibuja cajas (limita cantidad si hay muchas)
for (x,y,z,w,h,d) in items[:300]:
    draw_cuboid(ax, x,y,z,w,h,d, alpha=0.35)

# ejes
(xmin,xmax),(ymin,ymax),(zmin,zmax) = get_axis_limits(items, (W,H,D))
ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax); ax.set_zlim(zmin, zmax)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.view_init(elev=20, azim=35)
plt.title(f"Bin {BIN_TO_PLOT} — util={utilizations[BIN_TO_PLOT]:.3f}")
plt.show()


In [ ]:
# === Resumen por bin + listado + gráficas (barras y 3D) ===
import os
)

from deeppack3d import deeppack3d
import math
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# ---- Parámetros de corrida ----
METHOD = 'bl'              # heurística Bottom-Left
K = 40
DATA_MODE = 'file'         # usamos tu archivo de datos
PATH = "../data/input_worst_case.txt"  # asegúrate de que este archivo exista ahí
N_ITERS = 200              # solo se usa si DATA_MODE == 'generated'
SEED = 42                  # solo se usa si DATA_MODE == 'generated'

# (Opcional) tamaño del contenedor/pallet (W,H,D)
CONTAINER_SIZE = None      # None => se infiere por bounding box

# ---- Recolectar colocaciones por bin ----
bins = []               # lista de bins; cada bin es una lista de items
current_bin_items = None

# item_counter = ID de caja según orden de colocación
item_counter = 0

gen_kwargs = dict(data=DATA_MODE, verbose=0)
if DATA_MODE == 'file':
    gen_kwargs['path'] = PATH
else:
    gen_kwargs['n_iterations'] = N_ITERS
    gen_kwargs['seed'] = SEED

for result in deeppack3d(METHOD, K, **gen_kwargs):
    if result is None:
        # empieza un nuevo bin
        current_bin_items = []
        bins.append(current_bin_items)
        continue

    bin_id, (x, y, z), (w, h, d), meta = result

    if current_bin_items is None:
        current_bin_items = []
        bins.append(current_bin_items)

    item_counter += 1
    # Guardamos: (id_caja, x, y, z, w, h, d)
    current_bin_items.append((item_counter, x, y, z, w, h, d))

num_bins = len(bins)
print(f"🧱 Bins detectados: {num_bins}")

# ---- Listado de cajas por bin ----
for b_idx, items in enumerate(bins):
    print(f"\n=== BIN {b_idx} ===")
    if not items:
        print("  (sin cajas)")
        continue
    for (cid, x, y, z, w, h, d) in items:
        print(f"  Caja {cid:4d}: pos=({x:4.1f}, {y:4.1f}, {z:4.1f})  size=({w:4.1f}, {h:4.1f}, {d:4.1f})")

# ---- Utilidad por bin ----
def bin_utilization(items, container_size=None):
    """Devuelve (utilizacion, used_volume, container_volume, (W,H,D) usado)"""
    if not items:
        return 0.0, 0.0, 0.0, (0, 0, 0)

    # items = (cid, x, y, z, w, h, d)
    used_volume = sum(w * h * d for (_, x, y, z, w, h, d) in items)

    if container_size is None:
        # inferir por bounding box de lo empacado
        max_x = max(x + w for (_, x, y, z, w, h, d) in items)
        max_y = max(y + h for (_, x, y, z, w, h, d) in items)
        max_z = max(z + d for (_, x, y, z, w, h, d) in items)
        W, H, D = max_x, max_y, max_z
    else:
        W, H, D = container_size

    container_volume = max(1, W * H * D)  # evita div/0
    return (used_volume / container_volume, used_volume, container_volume, (W, H, D))

utilizations = []
sizes_used = []
for i, items in enumerate(bins):
    u, used_v, cont_v, sz = bin_utilization(items, CONTAINER_SIZE)
    utilizations.append(u)
    sizes_used.append(sz)
    print(f"Bin {i}: util={u:.3f}  used_vol={used_v:.0f}  cont_vol={cont_v:.0f}  size={sz}")

# ---- Gráfico de barras: utilización por bin ----
plt.figure(figsize=(8, 4))
plt.bar(range(len(utilizations)), utilizations)
plt.xlabel("Bin")
plt.ylabel("Utilización (0–1)")
plt.title(f"Utilización por bin — {METHOD.upper()}  k={K}")
plt.ylim(0, 1.0)
plt.show()

# ---- 3D del bin que elijas ----
BIN_TO_PLOT = 40  # cambia el índice para ver otro bin

def draw_cuboid(ax, x, y, z, w, h, d, alpha=0.35):
    # Si por algún error vino una caja "vacía", no la dibujes
    if w <= 0 or h <= 0 or d <= 0:
        return

    X = [x, x + w]; Y = [y, y + h]; Z = [z, z + d]
    faces = [
        [(X[0], Y[0], Z[0]), (X[1], Y[0], Z[0]), (X[1], Y[1], Z[0]), (X[0], Y[1], Z[0])],
        [(X[0], Y[0], Z[1]), (X[1], Y[0], Z[1]), (X[1], Y[1], Z[1]), (X[0], Y[1], Z[1])],
        [(X[0], Y[0], Z[0]), (X[1], Y[0], Z[0]), (X[1], Y[0], Z[1]), (X[0], Y[0], Z[1])],
        [(X[0], Y[1], Z[0]), (X[1], Y[1], Z[0]), (X[1], Y[1], Z[1]), (X[0], Y[1], Z[1])],
        [(X[0], Y[0], Z[0]), (X[0], Y[1], Z[0]), (X[0], Y[1], Z[1]), (X[0], Y[0], Z[1])],
        [(X[1], Y[0], Z[0]), (X[1], Y[1], Z[0]), (X[1], Y[1], Z[1]), (X[1], Y[0], Z[1])],
    ]
    poly = Poly3DCollection(faces, edgecolor='k', alpha=alpha)
    ax.add_collection3d(poly)

def get_axis_limits(items, fallback=(1, 1, 1)):
    if not items:
        return (0, 1), (0, 1), (0, 1)
    max_x = max(x + w for (_, x, y, z, w, h, d) in items)
    max_y = max(y + h for (_, x, y, z, w, h, d) in items)
    max_z = max(z + d for (_, x, y, z, w, h, d) in items)
    return (0, max(max_x, fallback[0])), (0, max(max_y, fallback[1])), (0, max(max_z, fallback[2]))

items = bins[BIN_TO_PLOT] if BIN_TO_PLOT < num_bins else []
(W, H, D) = sizes_used[BIN_TO_PLOT] if BIN_TO_PLOT < num_bins else (1, 1, 1)

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')

# dibuja contenedor solo con líneas (sin Poly3DCollection para evitar el bug)
def draw_container_wire(ax, W, H, D):
    edges = [
        ((0, 0, 0), (W, 0, 0)), ((W, 0, 0), (W, H, 0)),
        ((W, H, 0), (0, H, 0)), ((0, H, 0), (0, 0, 0)),
        ((0, 0, D), (W, 0, D)), ((W, 0, D), (W, H, D)),
        ((W, H, D), (0, H, D)), ((0, H, D), (0, 0, D)),
        ((0, 0, 0), (0, 0, D)), ((W, 0, 0), (W, 0, D)),
        ((W, H, 0), (W, H, D)), ((0, H, 0), (0, H, D)),
    ]
    for (x1, y1, z1), (x2, y2, z2) in edges:
        ax.plot([x1, x2], [y1, y2], [z1, z2], linewidth=1.2)

# dibuja contenedor inferido o indicado
draw_container_wire(ax, W, H, D)

# dibuja cajas (limita cantidad si hay muchas)
for (cid, x, y, z, w, h, d) in items[:300]:
    draw_cuboid(ax, x, y, z, w, h, d, alpha=0.35)

# ejes
(xmin, xmax), (ymin, ymax), (zmin, zmax) = get_axis_limits(items, (W, H, D))
ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax); ax.set_zlim(zmin, zmax)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.view_init(elev=20, azim=35)
plt.title(f"Bin {BIN_TO_PLOT} — util={utilizations[BIN_TO_PLOT]:.3f}")
plt.show()



In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU detectada, memory growth activado.")
    except RuntimeError as e:
        print(e)
else:
    print("⚠ TensorFlow no detecta GPU, usando CPU.")

# === Entrenar RL + Resumen por bin + listado + gráficas (barras y 3D) ===
import os
import glob
import shutil

)

from deeppack3d import deeppack3d
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

# ------------------ CONFIGURACIÓN GENERAL ------------------ #

# Lookahead
K = 40

# Archivo de test
TEST_DATA_MODE = 'file'
TEST_PATH = "../data/input_worst_case.txt"

# Entrenamiento RL
DO_TRAIN = True              # pon False si ya tienes un modelo entrenado
TRAIN_DATA_MODE = 'generated'  # 'generated' o 'file' si quieres entrenar con tu archivo
TRAIN_N_ITERS = 10           # nº de bloques de entrenamiento (cada uno = 100 episodios)
BATCH_SIZE = 32
SEED = 42

# (Opcional) tamaño del contenedor/pallet (W,H,D)
CONTAINER_SIZE = None      # None => se infiere por bounding box

MODEL_DIR = "../models"
MODEL_PATH = os.path.join(MODEL_DIR, f'k={K}.h5')

# ------------------ FASE 1: ENTRENAMIENTO RL ------------------ #

if DO_TRAIN:
    print(f"\n=== FASE 1: Entrenando RL con lookahead={K} ===")

    # Guarda qué modelos .h5 existen antes de entrenar
    h5_before = set(glob.glob("*.h5"))

    train_kwargs = dict(
        data=TRAIN_DATA_MODE,
        verbose=1,
        seed=SEED,
        train=True,
        batch_size=BATCH_SIZE,
        n_iterations=TRAIN_N_ITERS
    )
    if TRAIN_DATA_MODE == 'file':
        train_kwargs['path'] = TEST_PATH  # o pon otra ruta de entrenamiento

    # Ejecuta entrenamiento (la propia librería dibuja curvas y guarda el modelo)
    for _ in deeppack3d('rl', K, **train_kwargs):
        pass

    # Detectar nuevos .h5 tras el entrenamiento
    h5_after = set(glob.glob("*.h5"))
    new_models = list(h5_after - h5_before)

    if not new_models:
        print("⚠ No se encontró ningún nuevo archivo .h5 tras el entrenamiento.")
    else:
        # Si hay varios nuevos, tomamos el más reciente
        new_models.sort(key=lambda f: os.path.getmtime(f))
        latest_model = new_models[-1]
        print(f"✅ Modelo entrenado detectado: {latest_model}")

        os.makedirs(MODEL_DIR, exist_ok=True)
        shutil.copy(latest_model, MODEL_PATH)
        print(f"✅ Copiado a {MODEL_PATH} (usado luego en modo test)\n")

else:
    print("\n=== FASE 1: Entrenamiento saltado (DO_TRAIN=False) ===")

# ------------------ FASE 2: TEST RL CON TU ARCHIVO ------------------ #

print(f"\n=== FASE 2: Test RL con archivo {TEST_PATH} ===")

METHOD = 'rl'                # ahora sí usamos RL entrenado
DATA_MODE = TEST_DATA_MODE
PATH = TEST_PATH

# ---- Recolectar colocaciones por bin ----
bins = []               # lista de bins; cada bin es una lista de items
current_bin_items = None

# item_counter = ID de caja según orden de colocación
item_counter = 0

gen_kwargs = dict(data=DATA_MODE, verbose=0, n_iterations=-1)  # -1 => episodios hasta agotar datos
if DATA_MODE == 'file':
    gen_kwargs['path'] = PATH
else:
    gen_kwargs['seed'] = SEED

for result in deeppack3d(METHOD, K, **gen_kwargs):
    if result is None:
        # empieza un nuevo bin
        current_bin_items = []
        bins.append(current_bin_items)
        continue

    bin_id, (x, y, z), (w, h, d), meta = result

    if current_bin_items is None:
        current_bin_items = []
        bins.append(current_bin_items)

    item_counter += 1
    # Guardamos: (id_caja, x, y, z, w, h, d)
    current_bin_items.append((item_counter, x, y, z, w, h, d))

num_bins = len(bins)
print(f"\n🧱 Bins detectados (TEST RL): {num_bins}")

# ---- Listado de cajas por bin ----
for b_idx, items in enumerate(bins):
    print(f"\n=== BIN {b_idx} ===")
    if not items:
        print("  (sin cajas)")
        continue
    for (cid, x, y, z, w, h, d) in items:
        print(f"  Caja {cid:4d}: pos=({x:4.1f}, {y:4.1f}, {z:4.1f})  size=({w:4.1f}, {h:4.1f}, {d:4.1f})")

# ---- Utilidad por bin ----
def bin_utilization(items, container_size=None):
    """Devuelve (utilizacion, used_volume, container_volume, (W,H,D) usado)"""
    if not items:
        return 0.0, 0.0, 0.0, (0, 0, 0)

    # items = (cid, x, y, z, w, h, d)
    used_volume = sum(w * h * d for (_, x, y, z, w, h, d) in items)

    if container_size is None:
        # inferir por bounding box de lo empacado
        max_x = max(x + w for (_, x, y, z, w, h, d) in items)
        max_y = max(y + h for (_, x, y, z, w, h, d) in items)
        max_z = max(z + d for (_, x, y, z, w, h, d) in items)
        W, H, D = max_x, max_y, max_z
    else:
        W, H, D = container_size

    container_volume = max(1, W * H * D)  # evita div/0
    return (used_volume / container_volume, used_volume, container_volume, (W, H, D))

utilizations = []
sizes_used = []
for i, items in enumerate(bins):
    u, used_v, cont_v, sz = bin_utilization(items, CONTAINER_SIZE)
    utilizations.append(u)
    sizes_used.append(sz)
    print(f"Bin {i}: util={u:.3f}  used_vol={used_v:.0f}  cont_vol={cont_v:.0f}  size={sz}")

# ---- Gráfico de barras: utilización por bin ----
plt.figure(figsize=(8, 4))
plt.bar(range(len(utilizations)), utilizations)
plt.xlabel("Bin")
plt.ylabel("Utilización (0–1)")
plt.title(f"Utilización por bin — {METHOD.upper()}  k={K}")
plt.ylim(0, 1.0)
plt.show()

# ---- 3D del bin que elijas ----
BIN_TO_PLOT = 0  # cambia el índice para ver otro bin

def draw_cuboid(ax, x, y, z, w, h, d, alpha=0.35):
    # Si por algún error vino una caja "vacía", no la dibujes
    if w <= 0 or h <= 0 or d <= 0:
        return

    X = [x, x + w]; Y = [y, y + h]; Z = [z, z + d]
    faces = [
        [(X[0], Y[0], Z[0]), (X[1], Y[0], Z[0]), (X[1], Y[1], Z[0]), (X[0], Y[1], Z[0])],
        [(X[0], Y[0], Z[1]), (X[1], Y[0], Z[1]), (X[1], Y[1], Z[1]), (X[0], Y[1], Z[1])],
        [(X[0], Y[0], Z[0]), (X[1], Y[0], Z[0]), (X[1], Y[0], Z[1]), (X[0], Y[0], Z[1])],
        [(X[0], Y[1], Z[0]), (X[1], Y[1], Z[0]), (X[1], Y[1], Z[1]), (X[0], Y[1], Z[1])],
        [(X[0], Y[0], Z[0]), (X[0], Y[1], Z[0]), (X[0], Y[1], Z[1]), (X[0], Y[0], Z[1])],
        [(X[1], Y[0], Z[0]), (X[1], Y[1], Z[0]), (X[1], Y[1], Z[1]), (X[1], Y[0], Z[1])],
    ]
    poly = Poly3DCollection(faces, edgecolor='k', alpha=alpha)
    ax.add_collection3d(poly)

def get_axis_limits(items, fallback=(1, 1, 1)):
    if not items:
        return (0, 1), (0, 1), (0, 1)
    max_x = max(x + w for (_, x, y, z, w, h, d) in items)
    max_y = max(y + h for (_, x, y, z, w, h, d) in items)
    max_z = max(z + d for (_, x, y, z, w, h, d) in items)
    return (0, max(max_x, fallback[0])), (0, max(max_y, fallback[1])), (0, max(max_z, fallback[2]))

items = bins[BIN_TO_PLOT] if BIN_TO_PLOT < num_bins else []
(W, H, D) = sizes_used[BIN_TO_PLOT] if BIN_TO_PLOT < num_bins else (1, 1, 1)

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(111, projection='3d')

# dibuja contenedor solo con líneas (sin Poly3DCollection para evitar el bug)
def draw_container_wire(ax, W, H, D):
    edges = [
        ((0, 0, 0), (W, 0, 0)), ((W, 0, 0), (W, H, 0)),
        ((W, H, 0), (0, H, 0)), ((0, H, 0), (0, 0, 0)),
        ((0, 0, D), (W, 0, D)), ((W, 0, D), (W, H, D)),
        ((W, H, D), (0, H, D)), ((0, H, D), (0, 0, D)),
        ((0, 0, 0), (0, 0, D)), ((W, 0, 0), (W, 0, D)),
        ((W, H, 0), (W, H, D)), ((0, H, 0), (0, H, D)),
    ]
    for (x1, y1, z1), (x2, y2, z2) in edges:
        ax.plot([x1, x2], [y1, y2], [z1, z2], linewidth=1.2)

# dibuja contenedor inferido o indicado
draw_container_wire(ax, W, H, D)

# dibuja cajas (limita cantidad si hay muchas)
for (cid, x, y, z, w, h, d) in items[:300]:
    draw_cuboid(ax, x, y, z, w, h, d, alpha=0.35)

# ejes
(xmin, xmax), (ymin, ymax), (zmin, zmax) = get_axis_limits(items, (W, H, D))
ax.set_xlim(xmin, xmax); ax.set_ylim(ymin, ymax); ax.set_zlim(zmin, zmax)
ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
ax.view_init(elev=20, azim=35)
plt.title(f"Bin {BIN_TO_PLOT} — util={utilizations[BIN_TO_PLOT]:.3f}")
plt.show()
